In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%writefile access.py
import pandas as pd
import os

DEFAULT_FOLDER = "/content/drive/MyDrive/rw-shared-2026/sheets/"
OUTPUT_PLOTS_FOLDER = "/content/drive/MyDrive/cellular_rwanda_results/output_plots"

# --- THE SPEED FIX: IN-MEMORY CACHE ---
# This stops Colab from opening the same Excel file 3 different times.
_DATA_CACHE = {}

def _ensure_path(file_name, folder=DEFAULT_FOLDER):
    path = os.path.join(folder, file_name)
    print(f"DEBUG: _ensure_path attempting to access: {path}") # Added debug line
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {file_name} at {path}")
    return path

def _read_excel_cached(path, sheet_name, **kwargs):
    """Hidden helper function that remembers what it has already loaded."""
    # Create a unique key for the file and how it was opened
    key = str(path) + str(sheet_name) + str(kwargs)

    if key not in _DATA_CACHE:
        # Only reads from the hard drive if it hasn't seen this file before
        _DATA_CACHE[key] = pd.read_excel(path, sheet_name=sheet_name, **kwargs)

    # Always returns a fresh copy so your functions don't accidentally overwrite each other
    return _DATA_CACHE[key].copy()

# --- 1. MAPPING & CORE DATA ---

def load_id_lookup(folder=DEFAULT_FOLDER):
    """FIXED: Maps Column 1 (Device ID) to Column 0 (Application Name)."""
    path = _ensure_path("content.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="down_notld", skiprows=1, header=None)
    return dict(zip(df[1], df[0]))

def load_joined_temporal_data(folder=DEFAULT_FOLDER):
    """Loads raw traces and engineers time/day features for analysis."""
    path = _ensure_path("tcp-udp-bw.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="ip_flow_bwhist_g_u_agg")

    cols = list(df.columns)
    df = df.rename(columns={
        cols[0]: 'time_idx',
        cols[1]: 'device',
        cols[2]: 'code',
        cols[3]: 'rat',
        cols[4]: 'val',
        cols[7]: 'timestamp'
    })

    # Process Timestamps & Features for the Diurnal/Seasonal plots
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['timestamp'] = df.groupby('time_idx')['timestamp'].ffill().bfill()
    df['hour'] = df['timestamp'].dt.hour
    df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5

    return df

# --- 2. LATENCY & DNS DATA ---

def load_rtt_data(folder=DEFAULT_FOLDER):
    """Loads latency benchmarks for different technologies."""
    path = _ensure_path("rtt.xlsx", folder)
    return {
        "WAN (Satellite)": _read_excel_cached(path, sheet_name="wan", usecols=[0,1], names=["bin","count"]),
        "3G (UMTS)": _read_excel_cached(path, sheet_name="lan_umts", usecols=[0,1], names=["bin","count"]),
        "Mixed/2G": _read_excel_cached(path, sheet_name="lan", usecols=[0,1], names=["bin","count"])
    }

def load_dns_metrics(folder=DEFAULT_FOLDER):
    """Loads the 152 million DNS records for Signaling Tax analysis."""
    path = _ensure_path("dns.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="dnshist.txt", usecols=[0, 1], names=["bin", "count"])
    return df.dropna()

# --- 3. NETWORK ECONOMICS (TRAFFIC VOLUMES) ---

def load_traffic_asymmetry(folder=DEFAULT_FOLDER):
    """Loads raw interface throughput for 1:10 asymmetry analysis."""
    path = _ensure_path("tcp-udp-bw.xlsx", folder)
    return _read_excel_cached(path, sheet_name="ip_flow_bwhist_g_u_agg", header=0)

# --- 4. APPLICATION PAYLOAD (CONTENT PROFILES) ---

def load_content_total(folder=DEFAULT_FOLDER):
    """Extracts the exact 12.97 TB identified payload from Row 0."""
    path = _ensure_path("content.xlsx", folder)
    df_total = _read_excel_cached(path, sheet_name="down_notld", header=None, nrows=1)
    return df_total.iloc[0, 2] / 1e12 # Result in TB

def load_content_breakdown(folder=DEFAULT_FOLDER):
    """Loads domain-level breakdown (e.g., Googlevideo share)."""
    path = _ensure_path("content.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="down_notld", skiprows=1, header=None)
    df.columns = ["domain", "type", "bytes", "percentage"]
    return df

# --- 5. DEVICE CENSUS & CORE NETWORK DATA ---

def load_device_census(folder=DEFAULT_FOLDER):
    """Loads the device type census from data.xlsx."""
    path = _ensure_path("data.xlsx", folder)
    # The summary indicates the 'types' sheet has 8 rows of device counts
    # Explicitly select the first 3 columns as the file seems to have an extra column
    df = _read_excel_cached(path, sheet_name="types", header=None, usecols=[0, 1, 2])
    df.columns = ["Device_Type", "Count", "Percentage"]
    return df

def load_google_edge_geography(folder=DEFAULT_FOLDER):
    """Loads the Google PoP/Router traffic data."""
    path = _ensure_path("google_routers.xlsx", folder)
    # The 'allip' sheet contains up/down bytes by continent
    df = _read_excel_cached(path, sheet_name="allip")
    return df

def load_gtpc_churn(folder=DEFAULT_FOLDER):
    """Loads the 20-minute GTP-C session lifecycle capture."""
    path = _ensure_path("rw.gtpc.xlsx", folder)
    # Load the request/response pairing sheet to see session lifecycle
    df = _read_excel_cached(path, sheet_name="request")
    return df

def load_device_brands(folder=DEFAULT_FOLDER):
    """Loads the top mobile brands from data.xlsx."""
    path = _ensure_path("data.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="topbrands", header=None)

    # FIX: Dynamically handle 3 columns (Brand, Count, Percentage)
    if len(df.columns) == 3:
        df.columns = ["Brand", "Count", "Percentage"]
    else:
        # Fallback just in case: grab the first two columns and name them
        df = df.iloc[:, :2].copy()
        df.columns = ["Brand", "Count"]

    return df

def load_wan_rtt(folder=DEFAULT_FOLDER):
    """Loads the Wide Area Network (WAN) latency data to find the Satellite signature."""
    path = _ensure_path("rtt.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="wan", header=None)

    # FIX: Safely grab only the first two columns, ignoring any extra metadata columns
    df = df.iloc[:, :2].copy()

    df.columns = ["bin", "count"] # bin is time in seconds, count is frequency
    return df

def load_qos_metrics(folder=DEFAULT_FOLDER):
    """Loads TCP QoS to analyze the 93% Out-of-Order rate on 2G."""
    path = _ensure_path("qos_g_u_device.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="tcp_flow_qos_g_u_device", header=0)
    return df

def load_top_countries(folder=DEFAULT_FOLDER):
    """Loads Top-K destination countries from tcp.xlsx."""
    path = _ensure_path("tcp.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="tcpflow_country_topk", header=0)
    return df

def load_dns_histogram(folder=DEFAULT_FOLDER):
    """Loads the DNS RTT histogram from dns.xlsx."""
    path = _ensure_path("dns.xlsx", folder)
    # The summary states this is a 201-row histogram
    df = _read_excel_cached(path, sheet_name="dnshist.txt", header=None)

    # Safely grab the first two columns (bin and count)
    df = df.iloc[:, :2].copy()
    df.columns = ["RTT_Bin", "Query_Count"]
    # Force to numeric
    df["RTT_Bin"] = pd.to_numeric(df["RTT_Bin"], errors='coerce')
    df["Query_Count"] = pd.to_numeric(df["Query_Count"], errors='coerce')
    return df.dropna()

def load_geoip_correction(folder=DEFAULT_FOLDER):
    """Loads the true vs MaxMind continent mapping."""
    path = _ensure_path("dest_ip.xlsx", folder)
    df = _read_excel_cached(path, sheet_name="true", header=0)
    # Ensure column names are lowercase for easier matching
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def load_bufferbloat_stats(folder=DEFAULT_FOLDER):
    """Loads SYN vs ALL TCP stats to visualize bufferbloat."""
    path = _ensure_path("tcp.xlsx", folder)
    # Load the clean SYN (handshake) stats
    syn_df = _read_excel_cached(path, sheet_name="tcpflow_stats_syn", header=None)
    syn_df = syn_df.iloc[:, :5].copy()
    syn_df.columns = ['time', 'device', 'dir', 'avg_rtt', 'p97_rtt']
    syn_df['type'] = 'Clean Handshake (SYN)'

    # Load the bloated ALL flow stats
    all_df = _read_excel_cached(path, sheet_name="tcpflow_stats_all_rtt", header=None)
    all_df = all_df.iloc[:, :5].copy()
    all_df.columns = ['time', 'device', 'dir', 'avg_rtt', 'p97_rtt']
    all_df['type'] = 'Heavy Data Flow (Bufferbloat)'

    return pd.concat([syn_df, all_df], ignore_index=True)

def load_content_data(folder=DEFAULT_FOLDER):
    """
    Loads the top domains for both downlink and uplink from content.xlsx.
    The original Cambridge data stores these in the 'down_notld' and 'up_notld' sheets.
    """
    try:
        path = _ensure_path("content.xlsx", folder)
        # Load Downlink Data
        dl_df = _read_excel_cached(path, sheet_name='down_notld', skiprows=1, header=None)
        # Select the 'domain' (index 0) and 'bytes' (index 2) columns
        dl_df = dl_df.iloc[:, [0, 2]]
        dl_df.columns = ['domain', 'bytes'] # Standardizing column names
        dl_df['direction'] = 'Downlink'

        # Load Uplink Data
        ul_df = _read_excel_cached(path, sheet_name='up_notld', skiprows=1, header=None)
        # Select the 'domain' (index 0) and 'bytes' (index 2) columns
        ul_df = ul_df.iloc[:, [0, 2]]
        ul_df.columns = ['domain', 'bytes'] # Standardizing column names
        ul_df['direction'] = 'Uplink'

        # Combine them for easy analysis
        combined_df = pd.concat([dl_df, ul_df], ignore_index=True)
        return combined_df

    except FileNotFoundError:
        print(f"⚠️ Could not find content.xlsx at {path}. Please check the path.")
        return None

Writing access.py


In [3]:
%%writefile assess.py
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import os
import access # Import access.py to get the OUTPUT_PLOTS_FOLDER

# Create an output folder for the LaTeX presentation images
os.makedirs(access.OUTPUT_PLOTS_FOLDER, exist_ok=True)

# =====================================================================
# MODULE 1: VOLUMETRIC ANALYSIS & SIGNALING OVERHEAD
# =====================================================================

def discover_network_stats(bw_df, content_total_tb, rtt_df):
    """Calculates ground truth metrics: Total Volume, Efficiency, and Latency."""
    raw_tb = bw_df[bw_df['code'].isin(['dt', 'dm'])]['val'].sum() / 1e12
    efficiency = (content_total_tb / raw_tb) * 100
    waste_tb = raw_tb - content_total_tb

    rtt_sorted = rtt_df.sort_values('bin')
    cdf = rtt_sorted['count'].cumsum() / rtt_sorted['count'].sum()
    median_ms = rtt_sorted.loc[(cdf >= 0.5).idxmax(), 'bin'] * 1000

    print("--- Discovery Complete ---")
    print(f"Identified Median RTT: {median_ms:.1f} ms")
    print(f"Identified Raw Volume: {raw_tb:.2f} TB")
    print(f"Discovered Overhead:   {waste_tb:.2f} TB ({100-efficiency:.1f}%)")

    return {"raw_tb": raw_tb, "content_tb": content_total_tb, "waste_tb": waste_tb, "efficiency_pct": efficiency, "median_rtt": median_ms}

def plot_volumetric_efficiency(stats):
    """Visualizes Volumetric Protocol Efficiency (Data Plane vs. Control Plane)."""
    fig = plt.figure(figsize=(6, 8))
    plt.bar('Network Load', stats['content_tb'], label='User Payload (Data Plane)', color='#27ae60', width=0.6)
    plt.bar('Network Load', stats['waste_tb'], bottom=stats['content_tb'], label='Protocol Overhead (Control Plane)', color='#c0392b', width=0.6)
    plt.title(f"Volumetric Efficiency: {stats['efficiency_pct']:.1f}%", fontsize=14, fontweight='bold')
    plt.ylabel("Traffic Volume (Terabytes)"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "volumetric_efficiency.png"), dpi=300)
    plt.show()

def plot_protocol_tax_breakdown(df):
    """Categorizes infrastructural overhead into upload/download control channels."""
    breakdown = df.groupby('code')['val'].sum() / 1e12
    labels = {'dm': 'Downlink Control', 'um': 'Uplink Control', 'dt': 'Downlink Payload', 'ut': 'Uplink Payload'}
    breakdown.index = [labels.get(x, x) for x in breakdown.index]
    fig = plt.figure(figsize=(7, 7))
    plt.pie(breakdown, labels=breakdown.index, autopct='%1.1f%%', colors=sns.color_palette("muted"))
    plt.title("Network Load: Payload vs. Control Plane Breakdown", fontweight='bold')
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "protocol_tax_breakdown.png"), dpi=300)
    plt.show()

def plot_traffic_asymmetry(df):
    """Investigates directional network load (Upload vs. Download capabilities)."""
    asym = df.groupby('code')['val'].sum() / 1e12
    labels = {'dt': 'Downlink Payload', 'ut': 'Uplink Payload', 'dm': 'Downlink Control', 'um': 'Uplink Control'}
    asym.index = [labels.get(x, x) for x in asym.index]
    fig = plt.figure(figsize=(8, 4))
    asym.plot(kind='barh', color=['#3498db', '#e74c3c', '#95a5a6', '#7f8c8d'])
    plt.title("Traffic Asymmetry: Directional Infrastructure Load", fontweight='bold')
    plt.xlabel("Volume (Terabytes)")
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "traffic_asymmetry.png"), dpi=300)
    plt.show()

def plot_application_signaling_profiles(df, top_n=8):
    """Analyzes signaling-to-payload ratios to identify protocol-inefficient applications."""
    known_df = df[df['domain'] != 'Other/System']
    app_stats = known_df.groupby(['domain', 'code'])['val'].sum().unstack(fill_value=0)
    app_stats['Payload (dt+ut)'] = app_stats.get('dt', 0) + app_stats.get('ut', 0)
    app_stats['Signaling (dm+um)'] = app_stats.get('dm', 0) + app_stats.get('um', 0)

    top_apps = app_stats.nlargest(top_n, 'Payload (dt+ut)')
    top_apps['Signaling Ratio'] = top_apps['Signaling (dm+um)'] / top_apps['Payload (dt+ut)']
    top_apps = top_apps.sort_values('Signaling Ratio', ascending=False)

    fig = plt.figure(figsize=(10, 6))
    sns.barplot(x=top_apps.index, y=top_apps['Signaling Ratio'], palette='Reds_r')
    plt.title("Application Overhead: Signaling Generated per Payload Byte", fontsize=14, pad=20, fontweight='bold')
    plt.ylabel("Signaling Bytes per 1 Payload Byte")
    plt.xlabel("")
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "application_signaling_profiles.png"), dpi=300)
    plt.show()

def analyze_rat_efficiency(df):
    """Compares volumetric efficiency across Radio Access Technologies (RAT)."""
    rat_stats = df.groupby(['rat', 'code'])['val'].sum().unstack(fill_value=0)
    rat_stats['Payload'] = rat_stats.get('dt', 0) + rat_stats.get('ut', 0)
    rat_stats['Signaling'] = rat_stats.get('dm', 0) + rat_stats.get('um', 0)
    rat_stats['Efficiency %'] = (rat_stats['Payload'] / (rat_stats['Payload'] + rat_stats['Signaling'])) * 100
    valid_rats = rat_stats[rat_stats['Payload'] > 0].copy()

    fig = plt.figure(figsize=(8, 5))
    sns.barplot(x=valid_rats.index, y=valid_rats['Efficiency %'], palette='viridis')
    plt.title("Technology Overhead: Volumetric Efficiency by RAT Generation", fontsize=14, pad=20, fontweight='bold')
    plt.xlabel("Radio Access Technology (RAT Code)")
    plt.ylabel("Payload Efficiency (%)")
    plt.ylim(0, 110)
    plt.grid(axis='y', alpha=0.3)

    for i, val in enumerate(valid_rats['Efficiency %']):
        plt.text(i, val + 2, f"{val:.1f}%", ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "rat_efficiency.png"), dpi=300)
    plt.show()

def plot_ghost_hour_tax(df):
    """Analyzes background network overhead during off-peak periods."""
    df['Time_Block'] = pd.cut(df['hour'], bins=[-1, 0, 5, 17, 22, 24],
                              labels=['Night', 'Off-Peak (1AM-5AM)', 'Day', 'Peak (6PM-10PM)', 'Late Night'])

    stats = df.groupby(['Time_Block', 'code'], observed=False)['val'].sum().unstack(fill_value=0)
    stats['Payload (GB)'] = (stats.get('dt', 0) + stats.get('ut', 0)) / 1e9
    stats['Signaling (GB)'] = (stats.get('dm', 0) + stats.get('um', 0)) / 1e9

    focus = stats.loc[['Off-Peak (1AM-5AM)', 'Peak (6PM-10PM)']].copy()
    focus['Signaling %'] = (focus['Signaling (GB)'] / (focus['Payload (GB)'] + focus['Signaling (GB)'])) * 100

    fig = plt.figure(figsize=(8, 5))
    x = np.arange(len(focus))
    width = 0.35
    plt.bar(x - width/2, focus['Payload (GB)'], width, label='User Payload', color='#27ae60')
    plt.bar(x + width/2, focus['Signaling (GB)'], width, label='Control Signaling', color='#c0392b')

    for i in range(len(focus)):
        plt.text(i + width/2, focus['Signaling (GB)'].iloc[i] + 50, f"{focus['Signaling %'].iloc[i]:.1f}%\nSignaling", ha='center', fontweight='bold')

    plt.title("Diurnal Overhead: Off-Peak vs. Peak Network Load", pad=20, fontsize=14, fontweight='bold')
    plt.xticks(x, focus.index)
    plt.ylabel("Total Volume (Gigabytes)")
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "ghost_hour_tax.png"), dpi=300)
    plt.show()

def plot_hourly_signaling_volatility(df):
    """Maps the hour-by-hour volatility of the Signaling-to-Payload ratio."""
    hourly = df.groupby(['hour', 'code'], observed=False)['val'].sum().unstack(fill_value=0)
    payload = hourly.get('dt', 0) + hourly.get('ut', 0)
    signaling = hourly.get('dm', 0) + hourly.get('um', 0)
    ratio = (signaling / (payload + signaling)) * 100

    plt.figure(figsize=(12, 5))
    plt.plot(ratio.index, ratio.values, marker='o', linestyle='-', color='#d35400', linewidth=3, markersize=8)
    plt.title("Control Plane Volatility: 24-Hour Signaling Ratio", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Hour of Day (24h)")
    plt.ylabel("Signaling as % of Total Network Volume")
    plt.xticks(range(24))
    plt.fill_between(ratio.index, ratio.values.min(), ratio.values, color='#e67e22', alpha=0.2)
    plt.axhline(ratio.mean(), color='black', linestyle='--', label=f"Average Signaling Ratio ({ratio.mean():.1f}%)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "hourly_signaling_volatility.png"), dpi=300)
    plt.show()

# =====================================================================
# MODULE 2: TEMPORAL & SPATIAL PAYLOAD DYNAMICS
# =====================================================================

def plot_application_dominance(content_df):
    """Shows the top drivers of raw payload volume."""
    df_plot = content_df.dropna(subset=['domain']).sort_values(by='bytes', ascending=False).head(10)
    fig = plt.figure(figsize=(10, 6))
    plt.barh(df_plot['domain'], df_plot['bytes']/1e9, color='#f39c12')
    plt.gca().invert_yaxis()
    plt.title("Network Payload Distribution: Top Application Domains", fontweight='bold')
    plt.xlabel("Volume (Gigabytes)")
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "application_dominance.png"), dpi=300)
    plt.show()

def get_useful_diurnal_data(df, top_n_to_filter=5):
    """Filters dataset to isolate human payload activity from system background noise."""
    useful_df = df[df['code'].isin(['ut', 'dt'])].copy()
    known = useful_df[useful_df['domain'] != 'Other/System']
    if not known.empty:
        top_apps = known.groupby('domain')['val'].sum().nlargest(top_n_to_filter).index.tolist()
        useful_df = useful_df[useful_df['domain'].isin(top_apps + ['Other/System'])]
    return useful_df

def plot_useful_diurnal(df, top_n=5):
    """Plots the 24-hour rhythmic cycle of application payloads."""
    clean_df = get_useful_diurnal_data(df, top_n_to_filter=top_n)
    df_down = clean_df[clean_df['code'] == 'dt'].copy()
    hourly_stats = df_down.groupby(['hour', 'domain'])['val'].mean().reset_index()
    hourly_stats['val_gb'] = hourly_stats['val'] / 1e9
    total_hourly = df_down.groupby('hour')['val'].mean() / 1e9

    fig = plt.figure(figsize=(12, 6))
    sns.lineplot(data=hourly_stats, x='hour', y='val_gb', hue='domain', palette='husl', linewidth=2.5)
    plt.plot(total_hourly.index, total_hourly.values, label='TOTAL PAYLOAD', color='black', linestyle='--', alpha=0.6, linewidth=1.5)
    plt.title(f"Diurnal Application Demand: The Network Rhythm", fontsize=14, fontweight='bold')
    plt.xlabel("Hour of Day (24h)")
    plt.ylabel("Average Volume (Gigabytes)")
    plt.xticks(range(24)); plt.grid(alpha=0.3)
    plt.legend(title="Application Domain", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "useful_diurnal.png"), dpi=300)
    plt.show()

def plot_intra_week_volatility(df):
    """Highlights intra-week demand volatility (Weekday vs. Weekend payload differences)."""
    useful_df = df[(df['code'].isin(['ut', 'dt'])) & (df['domain'] != 'Other/System')].copy()
    useful_df['day_type'] = useful_df['timestamp'].dt.dayofweek.map(lambda x: 'Weekend' if x >= 5 else 'Weekday')
    stats = useful_df.groupby(['hour', 'day_type'])['val'].mean().reset_index()

    fig = plt.figure(figsize=(12, 6))
    sns.lineplot(data=stats, x='hour', y=stats['val']/1e9, hue='day_type', linewidth=3, palette=['#3498db', '#e74c3c'])
    plt.title("Intra-Week Demand Volatility: Behavioral Load Shifts", pad=20, fontsize=14, fontweight='bold')
    plt.ylabel("Average Volume (Gigabytes)")
    plt.xlabel("Hour of Day (24h)")
    plt.xticks(range(24)); plt.grid(alpha=0.3)

    plt.figtext(0.5, 0.01,
                "Insight: Network load profiles exhibit stark elasticity between working days and leisure days,\n"
                "requiring dynamic resource provisioning across urban and residential base stations.",
                ha="center", fontsize=10, bbox={"facecolor":"#e74c3c", "alpha":0.1, "pad":5})

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "intra_week_volatility.png"), dpi=300)
    plt.show()

def categorize_spatial_nodes(df):
    """Clusters physical base stations by traffic volume to identify spatial infrastructure stress."""
    node_stats = df.groupby('device')['val'].agg(['sum', 'count']).reset_index()
    kmeans = KMeans(n_clusters=2, n_init=10, random_state=42).fit(np.log1p(node_stats[['sum']]))

    # Ensure standard mapping of clusters
    centers = kmeans.cluster_centers_
    urban_label = 1 if centers[1][0] > centers[0][0] else 0
    node_stats['Category'] = [r'Urban/High-Load Node' if c == urban_label else r'Rural/Low-Load Node' for c in kmeans.labels_]

    fig = plt.figure(figsize=(10, 6))
    sns.scatterplot(data=node_stats, x='device', y='sum', hue='Category', palette='Set1', alpha=0.6)
    plt.yscale('log')
    plt.title("Spatial Node Clustering: Infrastructure Load Distribution", fontweight='bold')
    plt.ylabel("Total Traffic Volume (Log Scale)")
    plt.xlabel("Unique Base Station / Node ID")
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "spatial_nodes_clustering.png"), dpi=300)
    plt.show()

def plot_domain_asymmetry(df, top_n=6):
    """Shows application consumption (Download) versus production (Upload)."""
    useful = df[(df['code'].isin(['dt', 'ut'])) & (df['domain'] != 'Other/System')]
    domain_stats = useful.groupby(['domain', 'code'])['val'].sum().unstack(fill_value=0) / 1e9
    domain_stats['Total'] = domain_stats['dt'] + domain_stats['ut']
    top_domains = domain_stats.nlargest(top_n, 'Total')

    fig = plt.figure(figsize=(10, 6))
    y = np.arange(len(top_domains))
    plt.barh(y, top_domains['dt'], color='#3498db', label='Downlink (Consumption)')
    plt.barh(y, -top_domains['ut'], color='#e74c3c', label='Uplink (Production)')

    plt.title("Domain Directionality: Traffic Consumption vs. Production", pad=20, fontsize=14, fontweight='bold')
    plt.yticks(y, top_domains.index)
    plt.xlabel("Volume (Gigabytes) - Note: Uplink plotted on negative axis")
    plt.axvline(0, color='black', linewidth=1)
    plt.legend()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "domain_asymmetry.png"), dpi=300)
    plt.show()

def plot_network_inequality_lorenz(df):
    """Measures spatial load inequality using a Lorenz curve."""
    node_totals = df[df['code'].isin(['dt', 'ut'])].groupby('device')['val'].sum().sort_values(ascending=False)
    cum_payload = node_totals.cumsum() / node_totals.sum() * 100
    cum_nodes = np.arange(1, len(node_totals) + 1) / len(node_totals) * 100

    cum_payload = np.insert(cum_payload.values, 0, 0)
    cum_nodes = np.insert(cum_nodes, 0, 0)

    plt.figure(figsize=(8, 6))
    plt.plot(cum_nodes, cum_payload, label='Network Load Lorenz Curve', color='#e74c3c', linewidth=3)
    plt.plot([0, 100], [0, 100], linestyle='--', color='grey', label='Line of Perfect Equality')

    idx_20_pct = np.abs(cum_nodes - 20).argmin()
    payload_at_20 = cum_payload[idx_20_pct]

    plt.axvline(20, color='black', linestyle=':', alpha=0.5)
    plt.axhline(payload_at_20, color='black', linestyle=':', alpha=0.5)
    plt.scatter([20], [payload_at_20], color='black', s=100, zorder=5)
    plt.text(25, payload_at_20 - 5, f"Top 20% of Nodes\ncarry {payload_at_20:.1f}% of Traffic", fontweight='bold')

    plt.title("Spatial Network Inequality: Payload Distribution", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Cumulative % of Network Nodes")
    plt.ylabel("Cumulative % of Total Payload")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.xlim(0, 100); plt.ylim(0, 100)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "network_inequality_lorenz.png"), dpi=300)
    plt.show()

def plot_app_monoculture_entropy(df):
    """Calculates Shannon Entropy to evaluate application diversity across base stations."""
    useful = df[(df['code'].isin(['dt', 'ut'])) & (df['domain'] != 'Other/System')].copy()
    node_domain_vol = useful.groupby(['device', 'domain'], observed=False)['val'].sum().unstack(fill_value=0)

    p = node_domain_vol.div(node_domain_vol.sum(axis=1), axis=0)
    entropy = -(p * np.log2(p.replace(0, np.nan))).sum(axis=1)

    plt.figure(figsize=(10, 5))
    sns.histplot(entropy.dropna(), bins=15, kde=True, color='#9b59b6')
    plt.title("Application Ecosystem: Shannon Entropy of Domain Diversity per Node", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Shannon Entropy Score (Low = Monoculture, High = High Diversity)")
    plt.ylabel("Number of Network Nodes")

    mean_entropy = entropy.mean()
    plt.axvline(mean_entropy, color='red', linestyle='--', label=f'Network Mean Entropy: {mean_entropy:.2f}')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "app_monoculture_entropy.png"), dpi=300)
    plt.show()

def identify_researcher_windows(df):
    """Recommends the statistically optimal window for bulk academic data transfers."""
    useful_hourly = df[df['domain'] != 'Other/System'].groupby('hour')['val'].mean()
    best_hour = useful_hourly.rolling(window=4).mean().idxmin()
    print("\n--- STRATEGIC RECOMMENDATION ---")
    print(f"Optimal Network Utilization Window: {best_hour-4:02d}:00 to {best_hour:02d}:00")
    print("Benefit: Leverages off-peak capacity to avoid active congestion interference.")

# =====================================================================
# MODULE 3: CORE NETWORK HEALTH & SATURATION
# =====================================================================

def identify_node_saturation(df):
    """Identifies highly saturated base stations suffering from micro-outages."""
    node_hr = df.groupby(['device', 'hour', 'code'])['val'].sum().unstack(fill_value=0)
    node_hr['Payload'] = node_hr.get('dt', 0) + node_hr.get('ut', 0)
    node_hr['Signaling'] = node_hr.get('dm', 0) + node_hr.get('um', 0)
    node_hr['Total'] = node_hr['Payload'] + node_hr['Signaling']

    crashes = node_hr[(node_hr['Signaling'] / node_hr['Total'] > 0.90) & (node_hr['Total'] > 1e9)]
    print("\n--- INFRASTRUCTURE FINDING: BASE STATION SATURATION ---")
    print(f"Discovered {len(crashes)} discrete 'Micro-Outage' events.")
    print("These denote node-hours where control plane operations consumed >90% of capacity, failing to deliver user payload.")
    if not crashes.empty:
        top_crashes = crashes.reset_index().groupby('hour').size()
        print(f"Highest vulnerability time frame: {top_crashes.idxmax():02d}:00 (Frequency: {top_crashes.max()})")

def plot_anatomy_of_a_crash(df):
    """Profiles a specific base station suffering from a control plane collapse."""
    node_hr = df.groupby(['device', 'hour', 'code'])['val'].sum().unstack(fill_value=0)
    node_hr['Payload'] = node_hr.get('dt', 0) + node_hr.get('ut', 0)
    node_hr['Signaling'] = node_hr.get('dm', 0) + node_hr.get('um', 0)
    node_hr['Total'] = node_hr['Payload'] + node_hr['Signaling']
    node_hr['Signaling_Ratio'] = node_hr['Signaling'] / node_hr['Total']

    active_nodes = node_hr[node_hr['Total'] > 1e8]
    if active_nodes.empty:
        return

    worst_idx = active_nodes['Signaling_Ratio'].idxmax()
    worst_node = worst_idx[0]
    max_ratio = active_nodes.loc[worst_idx, 'Signaling_Ratio']

    node_data = node_hr.loc[worst_node].reset_index()

    fig = plt.figure(figsize=(10, 5))
    plt.plot(node_data['hour'], node_data['Signaling']/1e6, label='Control Plane (Signaling MB)', color='#c0392b', linewidth=3)
    plt.plot(node_data['hour'], node_data['Payload']/1e6, label='Data Plane (Payload MB)', color='#27ae60', linewidth=3)
    plt.fill_between(node_data['hour'], node_data['Signaling']/1e6, node_data['Payload']/1e6,
                     where=(node_data['Signaling'] > node_data['Payload']), color='red', alpha=0.15, label="Saturation State")

    plt.title(f"Node Saturation Profile: Control Plane Collapse (Node {worst_node})", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Hour of Day")
    plt.ylabel("Hourly Volume (Megabytes)")
    plt.xticks(range(24)); plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, f"anatomy_of_crash_node_{worst_node}.png"), dpi=300)
    plt.show()

def plot_tcp_ack_starvation(df):
    """Maps the correlation between excessive uplink signaling and impaired downlink throughput."""
    stats = df.groupby(['device', 'hour', 'code'])['val'].sum().unstack(fill_value=0)
    stats = stats[(stats.get('dt', 0) > 1e6) & (stats.get('um', 0) > 1e6)].copy()
    stats['Uplink_Signaling_MB'] = stats['um'] / 1e6
    stats['Downlink_Payload_GB'] = stats['dt'] / 1e9

    fig = plt.figure(figsize=(10, 6))
    sns.scatterplot(x=stats['Uplink_Signaling_MB'], y=stats['Downlink_Payload_GB'], alpha=0.5, color='#8e44ad', edgecolor=None)

    threshold = stats['Uplink_Signaling_MB'].quantile(0.90)
    plt.axvspan(threshold, stats['Uplink_Signaling_MB'].max(), color='red', alpha=0.1, label="Protocol Starvation Threshold (Top 10%)")

    plt.title("TCP Layer Impairment: Downlink Throughput vs. Uplink Control Congestion", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Uplink Control Plane Load (Megabytes)")
    plt.ylabel("Downlink User Payload Volume (Gigabytes)")
    plt.xscale('log'); plt.yscale('log')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "tcp_ack_starvation.png"), dpi=300)
    plt.show()

def plot_core_network_churn(gtpc_df):
    """Analyzes the rapid lifecycle churn of control plane contexts inside the Core Network."""
    type_col = [c for c in gtpc_df.columns if 'type' in str(c).lower() or 'msg' in str(c).lower()][0]
    msg_counts = gtpc_df[type_col].value_counts()

    type_map = {16: 'Create Session', 18: 'Update Session', 26: 'Delete Session'}
    msg_counts.index = msg_counts.index.map(lambda x: type_map.get(x, f"Type {x}"))

    plt.figure(figsize=(8, 5))
    plt.plot(msg_counts.index, msg_counts.values, marker='o', markersize=12, linewidth=3, color='#8e44ad')
    plt.fill_between(msg_counts.index, 0, msg_counts.values, alpha=0.2, color='#8e44ad')

    plt.title("Core Network Volatility: GTP-C Session Churn", pad=20, fontsize=14, fontweight='bold')
    plt.ylabel("Number of Control Plane Messages")
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "core_network_churn.png"), dpi=300)
    plt.show()

def analyze_cellular_intelligence(df):
    """Shows the macroscopic ratio of modern vs. legacy network utilization."""
    rat_volume = df.groupby('rat')['val'].sum() / 1e12
    fig = plt.figure(figsize=(7,7))
    plt.pie(rat_volume, labels=[f"RAT Code {int(i)}" for i in rat_volume.index], autopct='%1.1f%%')
    plt.title("Infrastructure Utilization: 2G vs 3G Load Distribution", fontweight='bold')
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "cellular_intelligence.png"), dpi=300)
    plt.show()

# =====================================================================
# MODULE 4: HARDWARE ECOSYSTEM & EDGE ROUTING
# =====================================================================

def plot_device_ecosystem(device_df):
    """Profiles the hardware classes actively connected to the cellular network."""
    plt.figure(figsize=(10, 6))
    str_cols = device_df.select_dtypes(include=['object']).columns
    name_col = str_cols[0] if len(str_cols) > 0 else device_df.columns[0]
    num_cols = device_df.select_dtypes(exclude=['object']).columns
    count_col = device_df[num_cols].max().idxmax()

    plot_df = device_df.sort_values(by=count_col, ascending=True)
    bars = plt.barh(plot_df[name_col].astype(str), plot_df[count_col], color='#2980b9')

    total_devices = plot_df[count_col].sum()
    for bar in bars:
        width = bar.get_width()
        percentage = (width / total_devices) * 100
        plt.text(width + (plot_df[count_col].max() * 0.02), bar.get_y() + bar.get_height()/2,
                 f"{percentage:.1f}%", va='center', fontweight='bold')

    plt.title("Hardware Topology: User Equipment Distribution", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Total Associated Devices (Log Scale)")
    plt.ylabel("Device Archetype")
    plt.xscale('log')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "device_ecosystem.png"), dpi=300)
    plt.show()

def plot_brand_dominance_and_dns(brand_df):
    """Maps equipment manufacturer market dominance to explain background DNS anomalies."""
    plt.figure(figsize=(10, 6))
    brand_df['Count'] = pd.to_numeric(brand_df['Count'], errors='coerce')
    brand_df = brand_df.dropna(subset=['Count'])

    str_col = brand_df.columns[0]
    brand_df = brand_df[~brand_df[str_col].astype(str).str.upper().isin(['ALL', 'TOTAL', 'UNKNOWN', 'NAN'])].copy()

    top_brands = brand_df.nlargest(10, 'Count').sort_values(by="Count", ascending=True)
    bars = plt.barh(top_brands[str_col].astype(str), top_brands['Count'], color='#d35400')

    total_devices = brand_df['Count'].sum()
    for bar in bars:
        width = bar.get_width()
        pct = (width / total_devices) * 100
        plt.text(width + (brand_df['Count'].max() * 0.01), bar.get_y() + bar.get_height()/2,
                 f"{pct:.1f}%", va='center', fontweight='bold')

    plt.title("Hardware Supply Chain: Original Equipment Manufacturer (OEM) Dominance", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Total Associated Devices")
    plt.ylabel("OEM Brand")

    plt.figtext(0.5, 0.01,
                "Insight: Market concentration of specific regional manufacturers strictly dictates hardware firmware configurations,\n"
                "directly influencing baseline DNS routing anomalies across the national grid.",
                ha="center", fontsize=10, bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "brand_dominance.png"), dpi=300)
    plt.show()

def plot_global_routing_cdn_dominance(country_df):
    """Visualizes macroscopic global IP routing to highlight CDN centralization."""
    plt.figure(figsize=(10, 6))
    str_col = country_df.select_dtypes(include=['object']).columns[0]
    clean_df = country_df[~country_df[str_col].astype(str).str.upper().isin(['ALL', 'TOTAL', 'UNKNOWN', 'NAN'])].copy()
    num_cols = clean_df.select_dtypes(include=['number']).columns
    byte_col = clean_df[num_cols].sum().idxmax()

    top_countries = clean_df.nlargest(10, byte_col).sort_values(by=byte_col, ascending=True)
    bars = plt.barh(top_countries[str_col].astype(str), top_countries[byte_col], color='#27ae60')

    plt.title("Global IP Routing: Content Delivery Network (CDN) Centralization", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Total Data Volume (Bytes)")
    plt.ylabel("Destination Geolocation")

    plt.figtext(0.5, 0.01,
                "Insight: High infrastructural dependency on external Western cloud hubs inherently introduces \n"
                "rigid speed-of-light propagation delays across submarine transit networks.",
                ha="center", fontsize=10, bbox={"facecolor":"#27ae60", "alpha":0.1, "pad":5})

    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "global_routing_cdn_dominance.png"), dpi=300)
    plt.show()

def plot_google_edge_routing(google_df):
    """Maps the physical geographic edge nodes servicing Google application traffic."""
    continent_col = [c for c in google_df.columns if 'cont' in str(c).lower()][0]
    down_col = [c for c in google_df.columns if 'down' in str(c).lower()][0]

    geo_stats = google_df.groupby(continent_col)[down_col].sum() / 1e9
    geo_stats = geo_stats.sort_values(ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(x=geo_stats.index, y=geo_stats.values, palette='magma')
    plt.title("Edge Infrastructure: Geographic Serving Origins for Major Content", pad=20, fontsize=14, fontweight='bold')
    plt.ylabel("Downlink Payload Delivered (Gigabytes)")
    plt.xlabel("Origin Continent")
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "google_edge_routing.png"), dpi=300)
    plt.show()

def plot_ip_geofiction(geo_df):
    """Highlights the error rate of Western IP mapping tooling when evaluating regional networks."""
    maxmind_col = next((c for c in geo_df.columns if 'maxmind' in str(c).lower() or 'mm' in str(c).lower()), None)
    correct_col = next((c for c in geo_df.columns if 'correct' in str(c).lower() or 'true' in str(c).lower()), None)

    if not maxmind_col or not correct_col:
        maxmind_col, correct_col = geo_df.columns[2], geo_df.columns[3]

    errors = geo_df[geo_df[maxmind_col] != geo_df[correct_col]].copy()
    error_counts = errors[maxmind_col].value_counts().head(5)

    plt.figure(figsize=(10, 6))
    bars = plt.bar(error_counts.index.astype(str), error_counts.values, color='#e67e22')
    plt.title("Geolocation Observability Impairment (IP Geofiction)", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Continent Erroneously Assigned by Database Tools")
    plt.ylabel("Count of Misclassified IP Addresses")

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + (error_counts.max()*0.02), int(yval), ha='center', fontweight='bold')

    plt.figtext(0.5, 0.01,
                "Insight: Industry-standard telemetry and IP location tools exhibit high failure rates in emerging markets, \n"
                "blinding automated network monitoring to true physical routing paths.",
                ha="center", fontsize=10, bbox={"facecolor":"#e67e22", "alpha":0.1, "pad":5})

    plt.grid(axis='y', alpha=0.3)
    plt.subplots_adjust(left=0.15, right=0.9, bottom=0.2, top=0.85)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "ip_geofiction.png"), dpi=300)
    plt.show()

# =====================================================================
# MODULE 5: PHYSICAL LIMITS & PROTOCOL MECHANICS
# =====================================================================

def profile_technology_benchmarks(segment_data_dict):
    """Outputs basic descriptive statistics for technology-specific latency."""
    print(f"\n{'Protocol/Tech':<15} | {'Median RTT':<12} | {'Peak Density'}")
    print("-" * 45)
    for label, df in segment_data_dict.items():
        df_sorted = df.sort_values('bin')
        cdf = df_sorted['count'].cumsum() / df_sorted['count'].sum()
        median_ms = df_sorted.loc[(cdf >= 0.5).idxmax(), 'bin'] * 1000
        peak_prob = (df['count'].max() / df['count'].sum()) * 100
        print(f"{label:<15} | {median_ms:>8.1f} ms | {peak_prob:>15.2f}%")

def plot_technology_signatures(df_dict):
    """Plots probability density functions for baseline network latency generation."""
    fig = plt.figure(figsize=(12, 6))
    colors = sns.color_palette("muted", len(df_dict))
    for (label, df), color in zip(df_dict.items(), colors):
        prob = (df['count'] / df['count'].sum()) * 100
        plt.plot(df['bin'] * 1000, prob, label=label, color=color, linewidth=2)
    plt.title("Physical Layer Latency: RTT Signatures by Protocol", fontsize=14, fontweight='bold')
    plt.xlabel("Round Trip Time (ms)")
    plt.ylabel("Probability Density (%)")
    plt.xlim(0, 1000); plt.legend(); plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "technology_signatures.png"), dpi=300)
    plt.show()

def plot_signaling_waterfall(discovered_median, redirect_rate=80):
    """Illustrates sequential connection latency penalties in early mobile networks."""
    values = [discovered_median, discovered_median, discovered_median * (redirect_rate / 100)]
    fig = plt.figure(figsize=(8, 5))
    bottom = 0
    colors = ['#9b59b6', '#8e44ad', '#663399']
    for label, val, color in zip(['DNS Resolution', 'TCP Handshake', 'HTTP Overhead'], values, colors):
        plt.bar('Connection Lifecycle', val, bottom=bottom, label=label, color=color, width=0.4)
        bottom += val
    plt.title(f"Cumulative Setup Latency (Baseline: {discovered_median:.0f} ms)", fontweight='bold')
    plt.ylabel("Aggregate Delay (ms)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "signaling_waterfall.png"), dpi=300)
    plt.show()

def plot_dns_penalty(dns_df):
    """Models the specific latency spike associated with physical radio link establishment."""
    dns_df['ms'] = dns_df['bin'] * 1000
    fig = plt.figure(figsize=(10, 4))
    plt.bar(dns_df['ms'], dns_df['count'], color='#9b59b6', width=50)
    plt.xlim(0, 2000)
    plt.title("Radio Resource Control (RRC): Wireless Link State Transition Latency", fontweight='bold')
    plt.xlabel("Establishment Delay (ms)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "dns_penalty.png"), dpi=300)
    plt.show()

def plot_satellite_backhaul_wall(wan_df):
    """Highlights structural speed-of-light limitations caused by GEO satellite routing."""
    wan_df['ms'] = wan_df['bin'] * 1000
    plot_df = wan_df[wan_df['ms'] <= 500]

    plt.figure(figsize=(10, 6))
    plt.plot(plot_df['ms'], plot_df['count'], color='#2c3e50', linewidth=2)
    plt.fill_between(plot_df['ms'], 0, plot_df['count'], color='#34495e', alpha=0.3)

    peak_ms = plot_df.loc[plot_df['count'].idxmax(), 'ms']
    plt.axvline(peak_ms, color='#e74c3c', linestyle='--', linewidth=2)
    plt.text(peak_ms + 10, plot_df['count'].max() * 0.9, f"Signal Propagation Wall\n({peak_ms:.0f}ms)", color='#e74c3c', fontweight='bold', fontsize=12)

    plt.title("Physical Infrastructure Mechanics: GEO Satellite Propagation Delay", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Wide Area Network (WAN) Round Trip Time (ms)")
    plt.ylabel("Observed Packet Frequency")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "satellite_backhaul_wall.png"), dpi=300)
    plt.show()

def plot_tcp_out_of_order_illusion(qos_df):
    """Analyzes transport layer failure triggered by Radio Link Control (RLC) retransmissions."""
    qos_df.columns = [str(c).strip().lower().replace(' ', '_') for c in qos_df.columns]
    dir_col = next((c for c in qos_df.columns if 'dir' in c), None)
    rat_col = next((c for c in qos_df.columns if 'rat' in c), 'rat')
    ooo_col = next((c for c in qos_df.columns if 'out' in c and 'order' in c), None)
    retrans_col = next((c for c in qos_df.columns if 'retrans' in c), None)

    dl_df = qos_df[qos_df[dir_col].astype(str).str.contains('dg|d', case=False, na=False)].copy() if dir_col else qos_df.copy()
    rat_map = {1: '2G (GPRS/EDGE)', 2: '2.5G (EDGE)', 3: '3G (UMTS)', 4: '3.5G (HSPA)'}

    dl_df[rat_col] = pd.to_numeric(dl_df[rat_col], errors='coerce')
    dl_df['Network_Gen'] = dl_df[rat_col].map(rat_map)
    dl_df = dl_df.dropna(subset=['Network_Gen'])

    dl_df['Has_OOO'] = (pd.to_numeric(dl_df[ooo_col], errors='coerce').fillna(0) > 0).astype(int)
    dl_df['Has_Retrans'] = (pd.to_numeric(dl_df[retrans_col], errors='coerce').fillna(0) > 0).astype(int)

    stats = dl_df.groupby('Network_Gen')[['Has_OOO', 'Has_Retrans']].mean() * 100
    stats = stats.reindex(['2G (GPRS/EDGE)', '2.5G (EDGE)', '3G (UMTS)', '3.5G (HSPA)']).dropna()

    plt.figure(figsize=(10, 6))
    x = np.arange(len(stats.index))
    width = 0.35

    bars1 = plt.bar(x - width/2, stats['Has_OOO'], width, label='Sequence Anomaly (Out-of-Order %)', color='#e74c3c')
    bars2 = plt.bar(x + width/2, stats['Has_Retrans'], width, label='TCP Congestion Trigger (Retransmission %)', color='#2c3e50')

    plt.title("Transport Protocol Inference Failure: 2G Radio-Layer Interference", pad=20, fontsize=14, fontweight='bold')
    plt.xticks(x, stats.index)
    plt.ylabel("Impacted Downlink Flows (%)")
    plt.legend()
    plt.grid(axis='y', alpha=0.3)

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width() / 2., height + 1, f"{height:.1f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.figtext(0.5, 0.01,
                "Insight: Physical 2G radio loss recovery mechanically forces out-of-order packet delivery, \n"
                "tricking upper-layer TCP into falsely diagnosing massive congestion and throttling bandwidth.",
                ha="center", fontsize=10, bbox={"facecolor":"#e74c3c", "alpha":0.1, "pad":5})

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "tcp_out_of_order_illusion.png"), dpi=300)
    plt.show()

def plot_dns_invisible_tax(dns_df):
    """Calculates the multiplicative delay impact of DNS lookup latency."""
    dns_df.columns = ["RTT_Bin", "Query_Count"]
    dns_df = dns_df.sort_values(by="RTT_Bin").copy()
    if dns_df["RTT_Bin"].max() <= 100:
        dns_df["RTT_Bin"] = dns_df["RTT_Bin"] * 1000

    total_queries = dns_df["Query_Count"].sum()
    dns_df['Cumulative_Pct'] = (dns_df["Query_Count"].cumsum() / total_queries) * 100

    plt.figure(figsize=(10, 6))
    plt.plot(dns_df["RTT_Bin"], dns_df['Cumulative_Pct'], color='#8e44ad', linewidth=3)
    plt.fill_between(dns_df["RTT_Bin"], 0, dns_df['Cumulative_Pct'], color='#9b59b6', alpha=0.2)

    plt.title("Protocol Overhead: The DNS Resolution Latency Long Tail", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Domain Name Resolution Time (ms)")
    plt.ylabel("Cumulative Proportion of Network Queries")

    plt.axhline(80, color='red', linestyle='--', alpha=0.7)
    plt.axvline(100, color='red', linestyle='--', alpha=0.7)
    plt.scatter([100], [80], color='red', s=100, zorder=5)
    plt.text(120, 77, "80% threshold\n(< 100ms)", color='red', fontweight='bold')

    plt.xlim(0, 1000); plt.ylim(0, 100); plt.grid(alpha=0.3)

    plt.figtext(0.5, 0.01,
                "Insight: Due to the high number of synchronous DNS queries required for modern web rendering, \n"
                "the 20% latency long-tail mathematically guarantees noticeable application stutter.",
                ha="center", fontsize=10, bbox={"facecolor":"#8e44ad", "alpha":0.1, "pad":5})

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "dns_invisible_tax.png"), dpi=300)
    plt.show()

def plot_bufferbloat_crisis(rtt_df):
    """Visualizes the extreme latency degradation caused by unmanaged Base Station buffer queues."""
    rtt_df['p97_rtt'] = pd.to_numeric(rtt_df['p97_rtt'], errors='coerce')
    rtt_df = rtt_df.dropna(subset=['p97_rtt']).copy()

    rtt_df['type'] = rtt_df['type'].replace({
        'Clean Handshake (SYN)': 'Connection Initiation\n(TCP SYN)',
        'Heavy Data Flow (Bufferbloat)': 'Active Payload Flow\n(Queue Degradation)'
    })

    stats = rtt_df.groupby('type')['p97_rtt'].mean().sort_values()

    plt.figure(figsize=(10, 6))
    bars = plt.barh(stats.index, stats.values, color=['#3498db', '#c0392b'])

    plt.title("Queue Management Mechanics: Cellular Bufferbloat Degradation", pad=20, fontsize=14, fontweight='bold')
    plt.xlabel("Average Latency at 97th Percentile (ms) [LOG SCALE]")
    plt.xscale('log')
    plt.xlim(10, stats.max() * 5)

    for bar in bars:
        width = bar.get_width()
        plt.text(width * 1.2, bar.get_y() + bar.get_height()/2, f"{int(width):,} ms", va='center', fontweight='bold')

    plt.figtext(0.5, 0.01,
                "Insight: Excessive, unmanaged memory buffers at physical cellular base stations hold packets in deep queues \n"
                "instead of dropping them, defeating TCP congestion control and effectively freezing user throughput.",
                ha="center", fontsize=10, bbox={"facecolor":"#c0392b", "alpha":0.1, "pad":5})

    plt.grid(axis='x', alpha=0.3)
    plt.subplots_adjust(left=0.25, right=0.85, bottom=0.2, top=0.85)
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "bufferbloat_crisis.png"), dpi=300)
    plt.show()

def analyze_legacy_traps(df):
    """
    DEEP EXTRACTION 6: Spatial Digital Redlining.
    Finds towers that are strictly confined to 2G/Legacy technologies.
    """
    # Map RAT codes to generations (Simplification for 2015 datasets: 1-2 is 2G, 3+ is 3G/HSPA)
    # 1: GPRS, 2: EDGE, 3: UMTS, 4: HSPA
    df['Tech_Gen'] = df['rat'].apply(lambda x: '3G_Capable' if x >= 3 else ('2G_Only' if x in [1, 2] else 'Unknown'))

    # Pivot to see which towers have 3G vs 2G data
    # We only look at known 2G/3G traffic
    tower_tech = df[df['Tech_Gen'] != 'Unknown'].groupby(['device', 'Tech_Gen'])['val'].sum().unstack(fill_value=0)

    # A tower is a "Legacy Trap" if it processes >0 bytes of 2G, but exactly 0 bytes of 3G
    legacy_traps = tower_tech[(tower_tech.get('2G_Only', 0) > 0) & (tower_tech.get('3G_Capable', 0) == 0)]
    hybrid_modern = tower_tech[tower_tech.get('3G_Capable', 0) > 0]

    trap_pct = (len(legacy_traps) / len(tower_tech)) * 100

    print(f"\n--- RESEARCH FINDING: DIGITAL REDLINING ---")
    print(f"Total Towers Analyzed: {len(tower_tech)}")
    print(f"Modern/Hybrid Towers (3G Capable): {len(hybrid_modern)}")
    print(f"Legacy Traps (Strictly 2G): {len(legacy_traps)} ({trap_pct:.1f}% of network)")
    print(f"Insight: {trap_pct:.1f}% of the physical network footprint is digitally redlining its users, capping them at 2G speeds where modern web protocols (like HTTPS and heavy TCP) will inherently fail via timeout.")

# The Academic Categorization Dictionary
domain_category_map = {
    'googlevideo': 'Streaming & Video', 'youtube': 'Streaming & Video', 'gvt1': 'Streaming & Video',
    'xvideos': 'Adult Content', 'pornhub': 'Adult Content',
    'facebook': 'Social Media', 'fbcdn': 'Social Media', 'instagram': 'Social Media', 'twitter': 'Social Media',
    'whatsapp': 'Chat & Messaging', 'viber': 'Chat & Messaging', 'skype': 'Chat & Messaging',
    'dropbox': 'Cloud Storage', 'drive': 'Cloud Storage',
    'google': 'Web Search & OS', 'googleapis': 'Web Search & OS', 'gstatic': 'Web Search & OS', 'akamaihd': 'CDN/Infrastructure'
}

def analyze_and_plot_content_categories(content_df):
    """
    PHASE 1 RESEARCH GAP: Content Type Classification.
    Categorizes raw domains and plots an academic-grade distribution chart.
    """
    if content_df is None:
        return

    # Apply Mapping
    df = content_df.copy()
    df['domain'] = df['domain'].astype(str).str.lower()
    df['Category'] = df['domain'].map(domain_category_map).fillna('Unclassified Web')

    # Aggregate by Category and Direction
    stats = df.groupby(['Category', 'direction'])['bytes'].sum().unstack(fill_value=0)

    # Convert to Terabytes (TB)
    stats = stats / 1e12
    stats['Total_TB'] = stats.sum(axis=1)

    # Sort for plotting
    stats = stats.sort_values(by='Total_TB', ascending=True)

    # --- ACADEMIC PLOTTING (Cambridge Style) ---
    plt.figure(figsize=(10, 6))

    # We use a stacked horizontal bar chart (monochrome/muted colors for academic print)
    plt.barh(stats.index, stats['Downlink'], color='#2c3e50', label='Downlink (TB)')
    plt.barh(stats.index, stats['Uplink'], left=stats['Downlink'], color='#7f8c8d', label='Uplink (TB)')

    plt.title("Traffic Composition by Content Category (Airtel Rwanda)", pad=15, fontsize=12, fontweight='bold')
    plt.xlabel("Total Data Volume (Terabytes)", fontsize=11)
    plt.ylabel("Content Category", fontsize=11)

    # Academic grid lines (subtle)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.legend(loc='lower right')

    plt.tight_layout()
    plt.savefig(os.path.join(access.OUTPUT_PLOTS_FOLDER, "content_category_distribution.png"), dpi=300)
    plt.show()

    print("--- Content Classification Complete ---")
    # Print the top 3 categories by total volume
    top_3 = stats.nlargest(3, 'Total_TB')
    print(f"Top Category: {top_3.index[0]} ({top_3['Total_TB'].iloc[0]:.2f} TB)")

Writing assess.py


## **Experiment 2: The 2015 Protocol-Level "Waterfall of Latency"**

**System Initialization & Data Ingestion**

In [ ]:
import access, assess, importlib
importlib.reload(access); importlib.reload(assess)
import os
os.makedirs(access.OUTPUT_PLOTS_FOLDER, exist_ok=True)

In [ ]:
print("Initializing Data Pipeline...")
# 1. Load the ID map and the exact payload volume
df_raw = access.load_joined_temporal_data()
# The id_map from access.load_id_lookup() was problematic, so we will generate a corrected one
content_df_for_mapping = access.load_content_breakdown()
id_to_domain_corrected = {idx: domain for idx, domain in content_df_for_mapping['domain'].dropna().items()}

# Apply the correct domain mapping to df_raw
df_raw['domain'] = df_raw['device'].map(id_to_domain_corrected).fillna('Other/System')

total_payload_tb = access.load_content_total()

# 2. Load the main trace dataset (Uses the high-speed cache)
df = access.load_joined_temporal_data()
# Apply the correct domain mapping to df
df['domain'] = df['device'].map(id_to_domain_corrected).fillna('Other/System')
print(f"Data Loaded: {len(df):,} records.")

# 3. Load the physical latency parameters
dns_df = access.load_dns_metrics()
total_payload_tb = access.load_content_total()
rtt_dict = access.load_rtt_data()
content_df = access.load_content_breakdown()
print(f"Successfully loaded {len(df):,} network flow records.")

**Diagnosing Network Efficiency (The 55 TB Context)**

In [ ]:
print("Analyzing Volumetric Efficiency and Protocol Overhead...")

# 1. Discover the Ground Truth
stats = assess.discover_network_stats(df, total_payload_tb, rtt_dict["3G (UMTS)"])
assess.plot_signaling_waterfall(210)
assess.plot_volumetric_efficiency(stats)

# 2. Break down the 'Zombie' Data by Protocol Type
assess.plot_protocol_tax_breakdown(df)

# 3. NEW: Show the physical Infrastructure Asymmetry (Upload vs Download)
asym_df = access.load_traffic_asymmetry()
assess.plot_traffic_asymmetry(asym_df)

**Radio Access Network (RAN) Diagnostics**

In [ ]:
print("Analyzing Cellular Technology and Radio Latency...")

# 1. The Cellular Market Share and Signaling Intensity
assess.analyze_cellular_intelligence(df)

# 2. The Latency Distributions for 3G vs Satellite vs 2G
assess.profile_technology_benchmarks(rtt_dict)
assess.plot_technology_signatures(rtt_dict)

# 3. The Cellular "Warm-up" Delay
assess.plot_signaling_waterfall(stats['median_rtt'])
# 4. The radio warm up latency
assess.plot_dns_penalty(dns_df)

**Human Behavior & Seasonality**

In [ ]:
print("Mapping Behavioral Seasonality and Payload Dominance...")

# 1. Show the Top 10 Dominant Applications
assess.plot_application_dominance(content_df)


In [ ]:

# 2. Plot the standard 24-hour Diurnal Cycle
assess.plot_useful_diurnal(df)

In [ ]:
assess.plot_intra_week_volatility(df)

In [ ]:
# MODULE 1 & 2: BEHAVIOR & SPATIAL DYNAMICS
# ---------------------------------------------------------
print("\n[Modules 1 & 2] Human Behavior & Payload Elasticity...")
# Note: Ensure you have loaded your baseline payload dataframe here (e.g., df_main)
assess.plot_intra_week_volatility(df)
assess.plot_domain_asymmetry(df)
assess.plot_network_inequality_lorenz(df)

In [ ]:
# ---------------------------------------------------------
# MODULE 4: HARDWARE ECOSYSTEM & EDGE ROUTING
# ---------------------------------------------------------
print("\n[Module 4] Hardware Supply Chain: TECNO/ITEL Duopoly...")
brand_df = access.load_device_brands()
assess.plot_brand_dominance_and_dns(brand_df)

print("\n[Module 4] Global Routing: Cloud & CDN Dominance...")
country_df = access.load_top_countries()
assess.plot_global_routing_cdn_dominance(country_df)

print("\n[Module 4] IP Geofiction: MaxMind Geolocation Failures...")
geo_df = access.load_geoip_correction()
assess.plot_ip_geofiction(geo_df)

In [ ]:
# ---------------------------------------------------------
# MODULE 5: PHYSICAL LIMITS & PROTOCOL MECHANICS
# ---------------------------------------------------------
print("\n[Module 5] Physical Limits: The Satellite Wall...")
wan_df = access.load_wan_rtt()
assess.plot_satellite_backhaul_wall(wan_df)

print("\n[Module 5] Protocol Illusions: 2G TCP Out-Of-Order Failure...")
qos_df = access.load_qos_metrics()
assess.plot_tcp_out_of_order_illusion(qos_df)

print("\n[Module 5] The Protocol Long Tail: DNS Invisible Tax...")
dns_df = access.load_dns_histogram()
assess.plot_dns_invisible_tax(dns_df)

print("\n[Module 5] Queue Degradation: The Bufferbloat Crisis...")
rtt_df = access.load_bufferbloat_stats()
assess.plot_bufferbloat_crisis(rtt_df)


print("\n" + "="*70)
print(f"ALL PLOTS SUCCESSFULLY GENERATED AND EXPORTED TO: ./{access.OUTPUT_PLOTS_FOLDER}/")
print("="*70)

**Spatial Vulnerability (Machine Learning)**

In [ ]:
print("Applying Machine Learning to Spatial Geography...")

# Run K-Means to identify Urban vs Rural infrastructure nodes
spatial_registry = assess.categorize_spatial_nodes(df)

**Strategic Recommendations**

In [ ]:
print("Calculating Optimal Delivery Windows...")

# To address the ValueError caused by an empty 'useful_hourly' series,
# we need to ensure df['domain'] is correctly mapped.
# The 'id_map' variable from earlier in the notebook appears to be incorrect.
# We'll re-map 'df' using content_df for a proper device ID to domain mapping.
import access
content_df = access.load_content_breakdown()
id_to_domain_corrected = {idx: domain for idx, domain in content_df['domain'].dropna().items()}
df['domain'] = df['device'].map(id_to_domain_corrected).fillna('Other/System')

# Output the final recommendation for the partners
assess.identify_researcher_windows(df)

In [ ]:
print("--- PHASE 6: ADVANCED RESEARCH CONTRIBUTIONS ---")

# SAFEGUARD: Ensure the 'domain' column exists before running deep dives
if 'domain' not in df_raw.columns:
    print("Re-applying Domain Mapping for Research Analysis...")
    content_df_for_mapping = access.load_content_breakdown()
    id_to_domain_corrected = {idx: domain for idx, domain in content_df_for_mapping['domain'].dropna().items()}
    df_raw['domain'] = df_raw['device'].map(id_to_domain_corrected).fillna('Other/System')

# Research Contribution 1: Application-Specific Signaling Profiles
print("\n1. Generating The 'Chatty App' Index...")
assess.plot_application_signaling_profiles(df_raw)

In [ ]:

# Research Contribution 2: Technology Efficiency Comparison
print("\n2. Analyzing Efficiency by Radio Access Technology (RAT)...")
assess.analyze_rat_efficiency(df_raw)

In [ ]:
# Contribution 3: Control Plane Collapse & TCP Starvation
print("\n3A. Extracting Micro-Outages (Anatomy of a Crash)...")
assess.plot_anatomy_of_a_crash(df_raw)

print("\n3B. Mapping TCP ACK Starvation...")
assess.plot_tcp_ack_starvation(df_raw)

In [ ]:
print("--- PHASE 7: UNTOUCHED DATA DIMENSIONS ---")

# Safeguard mapping if df_raw isn't mapped
if 'domain' not in df_raw.columns:
    content_df_for_mapping = access.load_content_breakdown()
    id_to_domain_corrected = {idx: domain for idx, domain in content_df_for_mapping['domain'].dropna().items()}
    df_raw['domain'] = df_raw['device'].map(id_to_domain_corrected).fillna('Other/System')

# Contribution 4: The Ghost Hour
print("\n4. Analyzing Nighttime 'Ghost Hour' Background Chatter...")
assess.plot_ghost_hour_tax(df_raw)

# Contribution 5: Domain Asymmetry
print("\n5. Mapping Application Consumption vs. Production...")
assess.plot_domain_asymmetry(df_raw)

# Contribution 6: Digital Redlining (Legacy Traps)
print("\n6. Scanning Network for '2G-Only' Legacy Traps...")
assess.analyze_legacy_traps(df_raw)

In [ ]:
print("--- PHASE 8: INFORMATION THEORY & NETWORK ECONOMICS ---")

# Contribution 7: The Pareto Trap
print("\n7. Mapping Network Inequality (Lorenz Curve)...")
assess.plot_network_inequality_lorenz(df_raw)

# Contribution 8: Shannon Entropy
print("\n8. Calculating Domain Diversity via Shannon Entropy...")
assess.plot_app_monoculture_entropy(df_raw)

# Contribution 9: The Breathing Tax
print("\n9. Modeling 24-Hour Signaling Volatility...")
assess.plot_hourly_signaling_volatility(df_raw)

In [ ]:
print("--- PHASE 9: CORE NETWORK & DIGITAL GEOGRAPHY ---")

# 1. Device Ecosystem
print("\n10. Analyzing Hardware Ecosystem...")
device_df = access.load_device_census()
assess.plot_device_ecosystem(device_df)

# 2. Google Edge Geography
print("\n11. Mapping Content CDN Geography...")
google_df = access.load_google_edge_geography()
assess.plot_google_edge_routing(google_df)

# 3. GTP-C Core Network Churn
print("\n12. Analyzing Core Gateway Session Churn...")
gtpc_df = access.load_gtpc_churn()
assess.plot_core_network_churn(gtpc_df)

In [ ]:
print("--- PHASE 10: HARDWARE ECONOMICS & PHYSICS ---")

# 1. The Hardware Supply Chain (Baidu Anomaly)
print("\n13. Analyzing Device Brand Market Share...")
brand_df = access.load_device_brands()
assess.plot_brand_dominance_and_dns(brand_df)

# 2. The Satellite Wall
print("\n14. Hunting for the Satellite Backhaul Signature...")
wan_df = access.load_wan_rtt()
assess.plot_satellite_backhaul_wall(wan_df)

In [ ]:
print("--- PHASE 11: PROTOCOL ILLUSIONS & WIRELESS PHYSICS ---")

# 1. The TCP Out-of-Order Illusion
print("\n15. Analyzing TCP QoS and 2G Out-Of-Order Rates...")
qos_df = access.load_qos_metrics()
assess.plot_tcp_out_of_order_illusion(qos_df)

In [ ]:
print("--- PHASE 12: GLOBAL ROUTING & THE INVISIBLE TAX ---")

# 1. The Norway Proxy Anomaly
print("\n16. Mapping Global Destination Anomalies...")
country_df = access.load_top_countries()
assess.plot_global_routing_cdn_dominance(country_df)

# 2. The DNS Long Tail
print("\n17. Analyzing the DNS Resolution 'Long Tail'...")
dns_df = access.load_dns_histogram()
assess.plot_dns_invisible_tax(dns_df)

In [ ]:
# 1. Run the fixed Brand Dominance
brand_df = access.load_device_brands()
assess.plot_brand_dominance_and_dns(brand_df)

In [ ]:
print("--- PHASE 13: DIGITAL GEOFICTION & BUFFERBLOAT ---")

# 1. IP Geofiction
print("\n18. Analyzing MaxMind Geolocation Errors...")
geo_df = access.load_geoip_correction()
assess.plot_ip_geofiction(geo_df)


In [ ]:
# 2. Bufferbloat Crisis
print("\n19. Analyzing Bufferbloat (SYN vs Data Latency)...")
rtt_df = access.load_bufferbloat_stats()
assess.plot_bufferbloat_crisis(rtt_df)

In [ ]:
import os
import access

print(f"Listing contents of: {access.OUTPUT_PLOTS_FOLDER}")
if os.path.exists(access.OUTPUT_PLOTS_FOLDER):
    for filename in os.listdir(access.OUTPUT_PLOTS_FOLDER):
        print(filename)
else:
    print(f"The directory '{access.OUTPUT_PLOTS_FOLDER}' does not exist.")

In [ ]:
print("Loading Content Data...")
content_data = access.load_content_data()

print("\nCategorizing Domains and Generating Plot...")
assess.analyze_and_plot_content_categories(content_data)